# Lab: Regression Tree

*In this lab, we will build a **Regression Tree** model using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
> * **Business Context:** This project addresses a real-estate analytics scenario for a hypothetical company, **"Housewise Analytics,"** operating in the California housing market. California’s housing market is characterized by diverse neighborhoods, rapidly changing prices, and growing demand for data-driven decision-making among buyers, sellers, and realtors.
> * **Business Problem:** Stakeholders currently rely on broad averages or intuition, lacking a precise and explainable method to estimate home prices for specific neighborhoods and property characteristics. This uncertainty leads to lost sales opportunities, mispricing, and inefficient market decisions.
> * **Project Goal:** To build a **predictive model** that uses available district-level features (such as location, median income, number of rooms, and more) to accurately estimate median home values in California. The objective is to empower more confident and data-driven pricing decisions that benefit both buyers and sellers.

---

> ### 📝 2. Analytic Approach Report (Summary)
> *   **Problem Type:** *Supervised Regression* — the goal is to predict a **continuous numerical value** (median house value) from district-level features.
> *   **Model Selection:** For this lab, the candidate model is a **Regression Tree (Decision Tree Regressor)**. This approach was chosen because it can naturally handle non-linear relationships and feature interactions, which are common in housing data. (Note: In previous labs, simple and multiple linear regression were already explored.)
> *   **Evaluation Metrics:**
>     *   **R-squared ($R^2$) & Adjusted R-squared:** To measure how much of the variance in house values the model can explain, while adjusting for the number of features.
>     *   **RMSE (Root Mean Squared Error):** To provide a direct and interpretable measure of average prediction errors, using the original units of the target (house value in $100,000s).
>     *   **Feature Importance:** To identify which features (e.g., median income, rooms, latitude/longitude) have the largest impact on price predictions.
> 
> *   **Motivation for Model Selection:**  In a real-world analytic workflow, **Exploratory Data Analysis (EDA)** would precede model building. EDA (including scatterplots and residual analysis) often uncovers non-linear relationships between predictors and housing prices, motivating the use of regression trees. Regression trees can capture threshold, interaction, and piecewise patterns that linear models miss.
>
> ⚠️ **Caution:** For clarity and focus, this lab considers only a regression tree. In real-world projects, it is best practice to compare several modeling approaches (including advanced ensembles and regularized models) to select the most robust solution.

---

> ### 📝 3. Data Requirements Report (Summary)
> *   **Data Source:** The lab will use the **California Housing dataset** (available via scikit-learn), which includes data aggregated from 20,000+ block groups across California.
> *   **Features (Independent Variables):**  
>     * Median income (`MedInc`)  
>     * Average number of rooms (`AveRooms`)  
>     * Average number of bedrooms (`AveBedrms`)  
>     * Population (`Population`)  
>     * Average house occupancy (`AveOccup`)  
>     * Latitude (`Latitude`)  
>     * Longitude (`Longitude`)
> *   **Target (Dependent Variable):**  
>     * Median house value (`MedHouseVal`) expressed in $100,000s.
> *   **Granularity & Privacy:**  
>     * Each row represents aggregated statistics for a California census block group (district), not individual households. The dataset is fully anonymized and contains no sensitive or personally identifiable information (PII).

---

## Stage 4: Data Collection

As usual, we start with importing the necessary libraries and configuring them.

In [95]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import re


from sklearn.datasets import fetch_california_housing

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1. Extract
First, we extract the data from its original source (`scikit-learn`) and store it in an untouched, "raw" format in the `/data/raw` directory. This creates a perfect, versionable mirror of the source system at the time of collection.

In [96]:
# Extract data from the source system
raw_data = fetch_california_housing()
print(f"Data has been extracted from the source system as '{type(raw_data)}'")

Data has been extracted from the source system as '<class 'sklearn.utils._bunch.Bunch'>'


In [97]:
# Create a DataFrame with the original, untouched column names
raw_df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)
raw_df['price'] = raw_data.target

# Define and create the raw data directory
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the raw Data Frame
raw_data_path = raw_data_dir / "california_housing_raw_v1.csv"
raw_df.to_csv(raw_data_path, index=False)

print(f"Extracted dataset has {raw_df.shape[0]} samples and {raw_df.shape[1]} features, with {raw_df.isnull().values.sum()} missing values out of {raw_df.size}.")
print(f"Raw, untouched data has been saved to: {raw_data_path}\n")
raw_df.head()

Extracted dataset has 20640 samples and 9 features, with 0 missing values out of 185760.
Raw, untouched data has been saved to: ../data/raw/california_housing_raw_v1.csv



,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### 4.2. Transform
Now, we perform the **Transform** step. We load the raw data from `/data/raw/`, apply lightweight corrections like standardizing column names, and fix any obvious data quality issues.

In [98]:
# Load the raw data
interim_df = pd.read_csv(raw_data_path)

In [99]:
# Standardize column names to snake_case
def camel_to_snake(name):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return s2.lower()

interim_df.columns = [camel_to_snake(col) for col in interim_df.columns]

In [100]:
# Manually check the minimum and maximum values for all numarical features for any abnormalities
print("\n--- Min/Max Summary of Validated Numerical Data ---")
display(interim_df.describe().loc[['min', 'max']])


--- Min/Max Summary of Validated Numerical Data ---


,med_inc,house_age,ave_rooms,ave_bedrms,population,ave_occup,latitude,longitude,price
min,0.4999,1.0,0.846154,0.333333,3.0,0.692308,32.54,-124.35,0.14999
max,15.0001,52.0,141.909091,34.066667,35682.0,1243.333333,41.95,-114.31,5.00001


### 4.3. Load
Now we save the cleaned, transformed DataFrame to the `/data/interim` directory. This file now becomes the official, clean starting point for all analysis.

In [101]:
# Define and create the interim data directory
interim_data_dir = Path("../data/interim")
interim_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the transformed DataFrame
interim_data_path = interim_data_dir / "california_housing_interim_v1.parquet"
interim_df.to_parquet(interim_data_path, index=False)

### 4.4. Verification
Finally, let's perform a sanity check. We'll load the interim data we just created and verify its structure and content.

In [102]:
# Load the final interim data and verify its contents
cal_housing_df = pd.read_parquet(interim_data_path)
print(f"Verification successful. The following DataFrame is ready for analysis with {cal_housing_df.shape[0]} samples and {cal_housing_df.shape[1]} features:")
cal_housing_df.sample(5)

Verification successful. The following DataFrame is ready for analysis with 20640 samples and 9 features:


,med_inc,house_age,ave_rooms,ave_bedrms,population,ave_occup,latitude,longitude,price
13988,3.0990,30.0,5.688559,1.125000,2778.0,2.942797,34.83,-117.15,0.668
1648,2.9591,2.0,4.482808,1.012894,1606.0,2.300860,37.99,-121.96,2.101
2337,5.0864,4.0,6.964286,1.084416,1024.0,3.324675,36.83,-119.67,1.137
4570,1.5885,30.0,2.318063,1.031414,2188.0,2.863874,34.06,-118.27,1.542
11205,3.0288,37.0,4.275720,1.069959,719.0,2.958848,33.83,-117.91,1.614



---

> ### 📝 4. Data Collection Report (Summary)
> * **Extraction:** The California Housing dataset was retrieved from scikit-learn’s repository and saved in its original, untouched form as a raw `.csv` file for reproducibility and provenance.
> * **Transformation:**  
>     * All column names were standardized to snake_case for consistency and usability.
>     * Data quality checks revealed unusually high values in `ave_occup` (average occupancy), `ave_rooms` (average rooms), and `ave_bedrms` (average bedrooms). These potential outliers were **not** automatically removed during transformation, as they may represent legitimate cases such as hotels, group living environments, or unique properties. Determining their treatment is left for the data scientist at the EDA stage, based on domain knowledge and analytic objectives.
> * **Loading:** The standardized dataset was saved as a `.parquet` file in the `/data/interim` directory, providing the analysis-ready starting point for all subsequent work.
> * **Verification:** The interim DataFrame contains 20,640 rows and 9 features, with all variable names in snake_case and no missing values detected.

---

## Stage 5: Data Understanding
The goal of this stage is to conduct **Exploratory Data Analysis (EDA)** to develop a deep understanding of the data's content, quality, and structure. Through descriptive statistics and visualization, we will identify patterns, detect anomalies, and test initial hypotheses.

### 5.1. Preparation for Exploratory Data Analysis (EDA)
Let's perform final preparatory checks and minor adjustments to ensure the DataFrame is perfectly suited for the analysis ahead.